# 07. 과적합 완화 Reduced Walk-forward 모델

450개 일괄 집계 feature와 66,244-parameter MLP를 축소한다. 개발일의 walk-forward OOF만 이용해 core feature에서 `compact_trend → compact_momentum → compact_time`을 하나씩 복구하며, 10분 event-bucket sampling과 날짜 균등 weighted loss를 적용한다.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENT.md').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.reduced_walk_forward import run_reduced_experiment

assert 'envs/urban' in str(Path(sys.executable).resolve()), sys.executable
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 120)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'python: {sys.executable}')

PROJECT_ROOT: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
python: /home/user/anaconda3/envs/urban/bin/python


## 1. Reduced V2 전체 실행

각 fold의 학습은 `session·symbol·10분 bucket`에서 가장 이른 한 행만 사용한다. 각 날짜의 전체 loss weight가 같아지도록 row weight를 조정하며, hidden 32·dropout 0.2·AdamW weight decay 0.001·최대 15 epoch로 제한한다.

In [2]:
result = run_reduced_experiment(PROJECT_ROOT)
print(f"device: {result['device']}")
print(f"selected groups: {result['selected_groups']}")
print(f"features: {len(result['selected_features'])}, parameters: {result['parameter_count']:,}")

device: cuda
selected groups: ['absolute_price', 'candle', 'volatility', 'price_position', 'compact_trend']
features: 90, parameters: 3,044


## 2. OOF 순차 feature 복구 결정

후보는 현재 모델보다 전체 OOF 기대수익 Spearman이 0.005 이상 개선되고, 최악 날짜 Spearman 하락이 0.02 이하이며, multiclass log-loss 증가가 0.01 이하일 때만 채택한다. 100개 feature 상한도 동시에 적용한다.

In [3]:
display(result['feature_decisions'])
display(result['variant_summaries'])

,step,candidate_group,variant,feature_count,accepted,reason,candidate_return_spearman,candidate_worst_session_return_spearman,candidate_multiclass_logloss,candidate_tp_pr_auc,current_return_spearman,current_worst_session_return_spearman,current_multiclass_logloss,current_tp_pr_auc,spearman_improvement,worst_session_spearman_drop,multiclass_logloss_increase,passes_spearman,passes_worst_session,passes_logloss
0,0,BASE,absolute_price+candle+volatility+price_position,82,True,PREDECLARED_BASE,0.110500,0.061925,1.054747,0.167154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,compact_trend,absolute_price+candle+volatility+price_positio...,90,True,OOF_CRITERIA_PASS,0.126019,0.072065,1.056064,0.163987,0.110500,0.061925,1.054747,0.167154,0.015519,-0.010140,0.001317,True,True,True
2,2,compact_momentum,absolute_price+candle+volatility+price_positio...,96,False,OOF_CRITERIA_FAIL,0.099684,0.047774,1.070168,0.152649,0.126019,0.072065,1.056064,0.163987,-0.026335,0.024291,0.014104,False,False,False
3,3,compact_time,absolute_price+candle+volatility+price_positio...,96,False,OOF_CRITERIA_FAIL,0.100676,0.057253,1.071183,0.153999,0.126019,0.072065,1.056064,0.163987,-0.025343,0.014812,0.015118,False,True,False


,variant,feature_count,return_spearman,worst_session_return_spearman,multiclass_logloss,tp_pr_auc
0,absolute_price+candle+volatility+price_position,82,0.110500,0.061925,1.054747,0.167154
1,absolute_price+candle+volatility+price_positio...,90,0.126019,0.072065,1.056064,0.163987
2,absolute_price+candle+volatility+price_positio...,96,0.099684,0.047774,1.070168,0.152649
3,absolute_price+candle+volatility+price_positio...,96,0.100676,0.057253,1.071183,0.153999


## 3. Walk-forward fold와 확률·기대수익 지표

In [4]:
display(result['folds'])
display(result['metrics'])

,fold,fit_sessions,inner_validation_session,payoff_history_sessions,evaluation_session,variant,feature_count,best_epoch,temperature,inner_raw_logloss,inner_calibrated_logloss,train_rows_before_sampling,train_rows_after_sampling,inner_rows
0,1,"[session_2026-07-07, session_2026-07-08]",session_2026-07-09,"[session_2026-07-07, session_2026-07-08, sessi...",session_2026-07-10,absolute_price+candle+volatility+price_positio...,90,15,0.80,0.894449,0.877407,9668,1017,6699
1,2,"[session_2026-07-07, session_2026-07-08, sessi...",session_2026-07-10,"[session_2026-07-07, session_2026-07-08, sessi...",session_2026-07-13,absolute_price+candle+volatility+price_positio...,90,15,1.05,1.046319,1.045679,16367,1712,6859
2,3,"[session_2026-07-07, session_2026-07-08, sessi...",session_2026-07-13,"[session_2026-07-07, session_2026-07-08, sessi...",session_2026-07-14,absolute_price+candle+volatility+price_positio...,90,15,1.05,1.045395,1.044765,23226,2430,5959


,evaluation_group,session,samples,multiclass_logloss,multiclass_brier,accuracy,macro_pr_auc,tp_pr_auc,tp_roc_auc,tp_positive_rate,tp_ece,return_spearman,return_mae,top1_mean_actual_return,top5_mean_actual_return
0,oof,ALL,17465,1.056064,0.580154,0.565188,0.388302,0.163987,0.734982,0.068709,0.010500,0.126019,0.012389,-0.006317,-0.002104
1,reference_test,ALL,15180,0.982151,0.550257,0.587022,0.379962,0.125579,0.783371,0.045257,0.018443,0.135952,0.009917,-0.002990,-0.001181
2,oof,session_2026-07-10,6859,1.077262,0.582808,0.570491,0.384207,0.178883,0.739638,0.086456,0.039265,0.129751,0.013864,-0.008501,-0.002208
3,oof,session_2026-07-13,5959,1.052201,0.586335,0.565196,0.380273,0.178920,0.788153,0.057728,0.014498,0.072065,0.011059,-0.006027,-0.002488
4,oof,session_2026-07-14,4647,1.029731,0.568311,0.557349,0.426044,0.174527,0.808143,0.056596,0.023640,0.199398,0.011919,-0.002520,-0.000647
5,reference_test,session_2026-07-15,6629,1.132758,0.645161,0.482727,0.366499,0.133187,0.745306,0.057927,0.016488,0.115984,0.012387,-0.007023,-0.002185
6,reference_test,session_2026-07-16,4384,0.926365,0.504162,0.631615,0.410709,0.125265,0.761717,0.049954,0.012225,0.177832,0.010050,-0.000417,-0.000251
7,reference_test,session_2026-07-17,4167,0.801253,0.447775,0.706024,0.349037,0.108262,0.857656,0.020158,0.028095,0.059078,0.005845,0.000831,-0.000644


## 4. OOF threshold와 순차 백테스트

threshold는 OOF에서만 선택한다. `NO_VALID_THRESHOLD`이면 수익이 가장 나은 후보를 진단용으로만 저장하며 거래 가능 신호로 승인하지 않는다.

In [5]:
display(result['selected_threshold'])
display(result['backtest_metrics'])
display(result['session_metrics'])

,selected_threshold,selection_status,oof_mean_net_return,oof_portfolio_return,oof_profit_factor,oof_filled_trades,oof_profitable_session_share,oof_worst_session_mean_net_return
0,0.004426,NO_VALID_THRESHOLD,-0.001184,-0.004285,0.93432,36,0.333333,-0.005927


,signals_above_threshold,ten_minute_signal_clusters,order_attempts,filled_trades,tp_trades,sl_trades,timeout_trades,tp_precision_given_fill,mean_net_return_per_fill,median_net_return_per_fill,net_return_on_deployed_capital,return_on_initial_capital,total_net_pnl,profit_factor,max_drawdown,skipped_same_symbol,skipped_position_limit,skipped_cash_limit,threshold,ending_cash,evaluation_group,selection_status
0,42,39,42,36,13,19,4,0.361111,-0.001184,-0.032960,-0.001191,-0.004285,-42.849486,0.934320,-0.032972,0,0,0,0.004426,9957.150514,oof,NO_VALID_THRESHOLD
1,21,5,11,8,0,6,2,0.000000,-0.023129,-0.033103,-0.023135,-0.018513,-185.125638,0.124486,-0.021145,10,0,0,0.004426,9814.874362,reference_test,NO_VALID_THRESHOLD


,signals_above_threshold,ten_minute_signal_clusters,order_attempts,filled_trades,tp_trades,sl_trades,timeout_trades,tp_precision_given_fill,mean_net_return_per_fill,median_net_return_per_fill,net_return_on_deployed_capital,return_on_initial_capital,total_net_pnl,profit_factor,max_drawdown,skipped_same_symbol,skipped_position_limit,skipped_cash_limit,threshold,ending_cash,evaluation_group,session
0,21,20,21,17,5,10,2,0.294118,-0.005927,-0.032971,-0.005939,-0.010092,-100.922151,0.699305,-0.019251,0,0,0,0.004426,9899.077849,oof,session_2026-07-10
1,16,14,16,14,5,8,1,0.357143,-0.003615,-0.032980,-0.003609,-0.005051,-50.510364,0.821854,-0.013849,0,0,0,0.004426,9949.489636,oof,session_2026-07-13
2,5,5,5,5,3,1,1,0.600000,0.021748,0.046263,0.021725,0.010858,108.583029,4.267410,-0.003292,0,0,0,0.004426,10108.583029,oof,session_2026-07-14
3,17,3,8,6,0,5,1,0.000000,-0.029736,-0.033108,-0.029735,-0.017851,-178.510507,0.000000,-0.017851,9,0,0,0.004426,9821.489493,reference_test,session_2026-07-15
4,1,1,1,1,0,1,0,0.000000,-0.032960,-0.032960,-0.032960,-0.003294,-32.937351,0.000000,-0.003294,0,0,0,0.004426,9967.062649,reference_test,session_2026-07-16
5,3,1,2,1,0,0,1,0.000000,0.026341,0.026341,0.026341,0.002632,26.322220,inf,0.000000,1,0,0,0.004426,10026.322220,reference_test,session_2026-07-17


## 5. Reference-only 평가와 배포 판정

2026-07-15~17은 이번 구조 결정에 이미 참고된 날짜이므로 pristine test가 아니다. 결과 그룹을 `reference_test`로 명시하고, 성능이 좋아도 `PASS`를 허용하지 않으며 새로운 미래 날짜가 필요하다.

In [6]:
display(result['deployment'])
status = result['deployment'].iloc[0]['deployment_status']
display(Markdown(f'**최종 상태: `{status}`**'))

,selection_status,oof_eligible,reference_return_pass,reference_sessions,reference_profitable_sessions,reference_profitable_session_share,reference_worst_session_mean_net_return,deployment_status,fresh_test_available
0,NO_VALID_THRESHOLD,False,False,3,0,0.0,-0.03296,FAIL,False


**최종 상태: `FAIL`**

## 6. 저장 artifact

In [7]:
display(pd.DataFrame({'artifact': result['paths'].keys(), 'path': map(str, result['paths'].values())}))

,artifact,path
0,predictions,/home/user/urbandatalab/YSLee/data/stock_data/...
1,metrics,/home/user/urbandatalab/YSLee/data/stock_data/...
2,variant_metrics,/home/user/urbandatalab/YSLee/data/stock_data/...
3,variant_summaries,/home/user/urbandatalab/YSLee/data/stock_data/...
4,variant_folds,/home/user/urbandatalab/YSLee/data/stock_data/...
5,selected_folds,/home/user/urbandatalab/YSLee/data/stock_data/...
6,selected_history,/home/user/urbandatalab/YSLee/data/stock_data/...
7,feature_decisions,/home/user/urbandatalab/YSLee/data/stock_data/...
8,feature_schema,/home/user/urbandatalab/YSLee/data/stock_data/...
9,payoffs,/home/user/urbandatalab/YSLee/data/stock_data/...
